Protocolo vigente: una trayectoria de 100 generaciones de carácter evolutivo (algoritmo evolutivo poblacional), sin diseño base. Luz 20 m, alturas y secciones variables. Objetivos: masa mínima, utilización promedio máxima y desplazamiento Z mínimo. Criterio principal L/700; transferencia cronológica complementaria. Los resultados previos deben regenerarse tras el entrenamiento.


# 02 — Comparar predictores sin mezclar unidades

**Pregunta:** ¿qué modelos predicen mejor las tres respuestas y cuánto cuesta ajustarlos?

`make study` ejecuta el entrenamiento y la comparación con validación. Las referencias son media y Ridge; se agregan Random Forest, SVR, GPR, CatBoost, XGBoost, LightGBM y MLP. Este notebook muestra el procedimiento mediante los artefactos guardados.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from truss_research.evaluacion.informe_estudio import read_object
from truss_research.evaluacion.presentacion import notebook_context

context = notebook_context()
STUDY_DIR, REPORT_DIR, CONFIG = context.study_dir, context.report_dir, context.config
DATA_DIR, STATE, RECORDS = CONFIG.data_dir, context.state, context.records
display(
    Markdown(
        f"**Estudio:** `{STUDY_DIR.name}` · **etiqueta:** `{CONFIG.label}` · **informe:** `{REPORT_DIR.name}`"
    )
)
if CONFIG.label == "comprobacion":
    display(
        Markdown(
            "**Comprobación técnica breve: sus métricas no constituyen resultados de tesis.**"
        )
    )

## 1. Revisar configuración y selección

Cada transformador se ajusta solo con las filas de entrenamiento de su ajuste. Optuna dispone de 30 intentos por búsqueda, incluido Ridge; Dummy no usa búsqueda. Los cinco pliegues externos excluyen cada diseño tanto de selección de parámetros como de ajuste OOF, usando tres pliegues internos.

La búsqueda y las OOF se realizan una vez por familia con semilla 42. Las semillas finales comparten esos parámetros; no representan datasets independientes. El presupuesto completo es 1.440 intentos. La comprobación breve no permite concluir qué modelo es mejor.


In [ ]:
display(
    pd.Series(
        {
            "familias": CONFIG.models,
            "semillas": CONFIG.seeds,
            "pliegues_externos": CONFIG.folds,
            "pliegues_internos": CONFIG.nested_folds,
            "intentos_por_búsqueda": CONFIG.trials,
        }
    )
)
families = pd.read_csv(REPORT_DIR / "family_comparison.csv")
validation = families.loc[families["split"].eq("validation")].sort_values("mean")
winner = validation.iloc[0]["model"]
run = read_object(
    RECORDS[f"predictor/{winner}/{CONFIG.representative_seed}"] / "run.json"
)
display(
    pd.Series(run["config"]["parameters"], name=f"Parámetros efectivos de {winner}")
)
display(
    pd.Series(
        run["response_scale"],
        index=["mass_kg", "util_ratio", "z_disp_m"],
        name="Desviaciones de entrenamiento",
    )
)

## 2. Elegir la familia usando validación

El MAE mide cuánto se equivoca el modelo en promedio. Para comparar kg, ratios y metros, se divide el MAE de cada respuesta por su desviación de entrenamiento y se promedian las tres cantidades. Un valor menor es mejor; no es un porcentaje.

Se elige la familia con menor media entre semillas y se fija su semilla representativa predefinida. La dispersión entre pliegues no es un intervalo de confianza poblacional.


In [ ]:
display(validation[["model", "mean", "std", "count"]])
figure, axis = plt.subplots(figsize=(10, 4))
axis.bar(
    validation["model"], validation["mean"], yerr=validation["std"].fillna(0), capsize=3
)
axis.set(ylabel="MAE normalizado", title="Comparación en validación")
axis.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()
display(
    Markdown(
        f"La familia seleccionada por validación es **{winner}**, con semilla representativa **{CONFIG.representative_seed}**."
    )
)

## 3. Ver los errores en las unidades originales

MAE, RMSE, sesgo y percentil 95 permiten distinguir error típico, errores grandes y tendencia a sobreestimar o subestimar. R² queda indefinido si la respuesta del grupo es constante. Los resultados por categoría deben acompañarse de su cantidad de diseños.


In [ ]:
metrics = pd.read_csv(REPORT_DIR / "predictive_runs.csv")
display(
    metrics.loc[
        metrics["split"].eq("validation"),
        [
            "model",
            "seed",
            "target",
            "mae",
            "rmse",
            "r2",
            "bias",
            "p95_absolute_error",
            "count",
        ],
    ]
)
predictions = pd.read_csv(
    RECORDS[f"predictor/{winner}/{CONFIG.representative_seed}"]
    / "validation_predictions.csv"
)
figure, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for axis, target in zip(axes, ("mass_kg", "util_ratio", "z_disp_m"), strict=True):
    axis.scatter(predictions[target], predictions[f"pred_{target}"], alpha=0.6)
    bounds = [
        min(predictions[target].min(), predictions[f"pred_{target}"].min()),
        max(predictions[target].max(), predictions[f"pred_{target}"].max()),
    ]
    axis.plot(bounds, bounds, color="black", linestyle="--")
    axis.set(xlabel="Registro", ylabel="Predicción", title=target)
plt.tight_layout()
plt.show()
display(pd.read_csv(REPORT_DIR / "category_metrics.csv").query("split == 'validation'"))

## 4. Pérdidas pareadas y distancia al entrenamiento

Los intervalos remuestrean diseños 2.000 veces, promediando pérdidas entre semillas. Son descriptivos, condicionados a estos datos y modelos; no certifican superioridad poblacional. Una diferencia negativa indica menor error que la referencia.

La distancia usa transformaciones y regiones ajustadas solo con entrenamiento. El error por distancia ayuda a reconocer situaciones menos representadas, sin demostrar generalización a nuevas condiciones físicas.


In [ ]:
display(
    pd.read_csv(REPORT_DIR / "predictive_paired_errors.csv").query(
        "split == 'validation'"
    )
)
display(
    pd.read_csv(REPORT_DIR / "predictive_seed_variation.csv").query(
        "split == 'validation'"
    )
)
display(
    pd.read_csv(REPORT_DIR / "errors_by_distance.csv").query("split == 'validation'")
)
display(pd.read_csv(REPORT_DIR / "diagnostics.csv"))

## 5. Separar los costos

La búsqueda de hiperparámetros, las predicciones fuera de pliegue y el ajuste final tienen costos distintos. Los tiempos copiados en las filas de prueba no deben sumarse otra vez. La inferencia individual y por lote se miden por separado. No hay medición del tiempo del software estructural de origen.


In [ ]:
costs = pd.read_csv(REPORT_DIR / "predictive_costs.csv")
costs = costs.assign(
    nested_trial_count=costs.get(
        "nested_trial_count", pd.NA if CONFIG.nested_folds else 0
    )
)
display(
    costs.loc[
        costs["split"].eq("validation"),
        [
            "model",
            "seed",
            "trial_count",
            "nested_trial_count",
            "tuning_seconds",
            "crossfit_seconds",
            "fit_seconds",
            "individual_median_seconds",
            "batch_median_seconds",
        ],
    ]
)

## 6. Explicar con SHAP

SHAP usa una referencia de entrenamiento y casos de validación comunes. Expresa contribuciones en las unidades de cada respuesta. Si falla la comprobación de aditividad de árboles, se registra el motivo y se utiliza permutación.


In [ ]:
importance = pd.read_csv(REPORT_DIR / "shap_global.csv")
display(pd.read_csv(REPORT_DIR / "shap_costs.csv"))
families = pd.read_csv(REPORT_DIR / "family_comparison.csv")
winner = families.query("split == 'validation'").sort_values("mean").iloc[0]["model"]
display(importance.loc[importance["model"].eq(winner)])

### Importancia global: Comparar la importancia global

La media del valor absoluto SHAP resume cuánto contribuye cada entrada a las predicciones. No mide precisión ni permite comparar directamente magnitudes de respuestas con unidades distintas. Las entradas correlacionadas pueden compartir o redistribuir su importancia.


In [ ]:
selected = importance.loc[importance["model"].eq(winner)]
figure, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, target in zip(axes, ("mass_kg", "util_ratio", "z_disp_m"), strict=True):
    values = (
        selected.loc[selected["target"].eq(target)]
        .groupby("variable")["abs_shap"]
        .mean()
        .sort_values()
    )
    values.plot.barh(ax=axis)
    axis.set(title=target, xlabel="Media de |SHAP|")
plt.tight_layout()
plt.show()
display(pd.read_csv(REPORT_DIR / "shap_stability.csv"))

### Ejemplo local: Leer un ejemplo local

Un valor positivo aumenta la predicción respecto del valor base; uno negativo la reduce. Los ejemplos exportados son los tres identificadores menores en orden lexicográfico, iguales para todos los modelos. La tabla incluye los valores de entrada para contextualizar las contribuciones.


In [ ]:
examples = pd.read_csv(REPORT_DIR / "shap_examples.csv")
local = examples.loc[
    examples["model"].eq(winner) & examples["seed"].eq(CONFIG.representative_seed)
]
example_id = sorted(local["iter_id"].unique())[0]
display(
    local.loc[
        local["iter_id"].eq(example_id),
        ["iter_id", "target", "variable", "feature_value", "shap_value", "base_value"],
    ]
)
local.loc[local["iter_id"].eq(example_id)].pivot(
    index="variable", columns="target", values="shap_value"
).plot.bar(
    subplots=True, figsize=(9, 8), title=f"Contribuciones del diseño {example_id}"
)
plt.tight_layout()
plt.show()

## Conclusión

La selección usa validación y un criterio común. El error se interpreta por respuesta, cobertura y distancia. Las semillas pueden producir predicciones idénticas. SHAP describe asociaciones, sin demostrar causalidad o seguridad. Una predicción precisa tampoco garantiza una buena búsqueda.


## Precisión, estabilidad y utilidad

La comparación principal usa MAE normalizado; sus errores físicos y dispersión se reportan por respuesta. Las semillas comparten partición e hiperparámetros. La relación error–AUC usa ranking fijo, medias por familia y Spearman descriptivo; no considera cada semilla una nueva muestra estructural.


In [ ]:
from IPython.display import Image

display(pd.read_csv(REPORT_DIR / "prediction_utility.csv"))
display(pd.read_csv(REPORT_DIR / "prediction_utility_spearman.csv"))
for figure in sorted(REPORT_DIR.glob("prediction_utility_*.png")):
    display(Image(filename=str(figure)))
display(pd.read_csv(REPORT_DIR / "predictive_seed_dispersion.csv"))

## Diagnósticos de robustez
Los errores extremos se conservan y se presentan por separado. La cobertura escasa y la procedencia física pendiente limitan las conclusiones; no se retocan modelos a partir de prueba.

In [ ]:
from truss_research.evaluacion.robustez import EVIDENCE_NOTES

display(Markdown("\n\n".join(EVIDENCE_NOTES.values())))
for diagnostic in (
    "error_tail_summary.csv",
    "category_coverage.csv",
    "physical_provenance.csv",
    "predictive_seed_variation.csv",
    "convergence_diagnostics.csv",
):
    diagnostic_path = REPORT_DIR / diagnostic
    if diagnostic_path.is_file():
        display(pd.read_csv(diagnostic_path).head(20))

## Recursos computacionales y costo de uso

Se separan búsqueda y OOF del ajuste final y de la inferencia. La RSS es del proceso completo, con línea base; no equivale al tamaño del modelo. Las repeticiones temporales no aumentan el número de diseños.

In [ ]:
from IPython.display import Image

cost_root = REPORT_DIR / "benchmark"
if cost_root.is_dir():
    display(pd.read_csv(cost_root / "predictor_costs.csv"))
    display(pd.read_csv(cost_root / "research_cost.csv"))
    for figure_path in sorted(cost_root.glob("prediction_quality_cost.png")):
        display(Image(filename=str(figure_path)))
else:
    display(
        Markdown(
            "Medición pendiente: ejecutar make benchmark y después make report para este estudio."
        )
    )

## Intervalos y generalización

Los intervalos split conformal se calibran fuera del ajuste. Cobertura marginal nominal no significa cobertura condicional, conjunta ni garantizada durante selección adaptativa. La comparación por generaciones reentrena los predictores con purga; no constituye prueba externa independiente.


In [ ]:
for filename in [
    "targetwise_components.csv",
    "selected_interval_coverage.csv",
    "interval_summary.csv",
    "interval_by_web_config.csv",
    "interval_by_distance_region.csv",
    "chronological_selection.csv",
]:
    path = REPORT_DIR / filename
    if path.is_file():
        display(Markdown("**" + filename + "**"))
        display(pd.read_csv(path).head(30))

## Incertidumbre hacia generaciones posteriores

La calibración cronológica usa bloques de generaciones separados del ajuste. La cobertura se interpreta junto con amplitud, tamaño y categorías ausentes; no constituye una garantía estructural.

In [ ]:
for filename in ["chronological_coverage.csv", "chronological_generation_overlap.csv"]:
    path = REPORT_DIR / filename
    if path.is_file():
        display(pd.read_csv(path).head(30))

## TabPFN como familia elegible

TabPFN utiliza pesos preentrenados congelados y las semillas del estudio. Participa en la selección principal y en targetwise, sin Optuna ni fine-tuning. Puede alimentar RL y la selección adaptativa. Los intervalos utilizan calibración independiente.


In [ ]:
if CONFIG.tabpfn_enabled:
    fixed_metrics = pd.read_csv(REPORT_DIR / "predictive_runs.csv")
    display(fixed_metrics.loc[fixed_metrics.model.eq("tabpfn")])